# 18 · STM32 Hardware Comparison — MobileNetV2 vs MobileNetV3

**Purpose:** Deploy both student candidates on the NUCLEO-N657X0-Q and compare
real hardware performance. This is a standalone benchmark — not part of the
compression ablation (notebooks 13–16).

## What this notebook answers

| Question | How |
|----------|-----|
| Which model is faster on STM32? | Compare latency from X-CUBE-AI profiling |
| Does the 9× size gap translate to latency difference? | V2 (0.68MB) vs V3 (6.21MB) |
| Does quantization close the accuracy gap between them? | V2 78.80% vs V3 77.40% pre-quant |

## Models
- **MobileNetV2 scratch** — selected student (0.68MB FP32, ~0.17MB INT8)
- **MobileNetV3-Small pretrained** — best V3 candidate (6.21MB FP32, ~1.55MB INT8)

Both models are exported FP32 + INT8 QDQ ONNX and deployed to STM32.
Results feed into the hardware comparison section of notebook 17.

## Board
NUCLEO-N657X0-Q — STM32N657X0H3Q, Cortex-M55 @ 800MHz, 4.2MB SRAM


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, shutil, os
UTILS_SRC = "/content/drive/My Drive/thesis/utils"
if os.path.exists(UTILS_SRC):
    shutil.copytree(UTILS_SRC, "/content/utils", dirs_exist_ok=True)
    sys.path.insert(0, "/content")
    print("✅ utils loaded from Drive")
else:
    sys.path.insert(0, "/content")


In [ ]:
import os, time
import numpy as np
import torch
import torch.nn as nn

from utils.dataset import prepare_dataset, get_loaders, manifest_paths, VWWDataset, get_eval_transform
from utils.models  import MobileNetV2_Scratch, MobileNetV3_Pretrained, count_params, model_size_mb
from utils.train   import setup_device, evaluate

device = setup_device(seed=41)


In [ ]:
prepare_dataset()
_, val_loader = get_loaders(batch_size=64)


In [ ]:
SAVE_DIR = "/content/drive/My Drive/Colab Notebooks"


In [ ]:
# ── Load both models ─────────────────────────────────────────────────
# Update seed_XX values after running notebooks 06 and 09

V2_CKPT = f"{SAVE_DIR}/mobilenetv2_baseline_seed_XX.pth"   # from nb 06
V3_CKPT = f"{SAVE_DIR}/mobilenetv3_baseline_seed_XX.pth"   # from nb 09

model_v2 = MobileNetV2_Scratch().to(device)
model_v2.load_state_dict(torch.load(V2_CKPT, map_location=device))
model_v2.eval()

model_v3 = MobileNetV3_Pretrained().to(device)
model_v3.load_state_dict(torch.load(V3_CKPT, map_location=device))
model_v3.eval()

for name, m in [("MobileNetV2 scratch", model_v2), ("MobileNetV3 pretrained", model_v3)]:
    acc      = evaluate(m, val_loader, device)
    tp, _    = count_params(m)
    size     = model_size_mb(m)
    print(f"{name:30s}  val={acc*100:.2f}%  params={tp/1e6:.3f}M  size={size:.2f}MB")


In [ ]:
!pip -q install onnx onnxruntime onnxscript
import onnx
import onnxruntime as ort
from onnxruntime.quantization import (
    quantize_static, QuantType, QuantFormat, CalibrationDataReader
)
from torch.utils.data import DataLoader
from pathlib import Path

EXPORT_DIR    = Path(f"{SAVE_DIR}/exports"); EXPORT_DIR.mkdir(exist_ok=True)
IMG_SIZE      = 96
CALIB_SAMPLES = 200
EVAL_SAMPLES  = 200

def export_fp32_onnx(model, tag):
    path  = str(EXPORT_DIR / f"vww_{tag}_fp32.onnx")
    dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
    torch.onnx.export(
        model.cpu(), dummy, path,
        input_names=["input"], output_names=["logits"],
        opset_version=18, do_constant_folding=True,
        dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
        dynamo=False,
    )
    onnx.checker.check_model(path)
    model.to(device)
    print(f"✅ FP32 ONNX: {path}")
    return path

def collect_arrays(n_calib, n_eval):
    mp       = manifest_paths()
    eval_tf  = get_eval_transform()
    train_ds = VWWDataset(mp["train_person"], mp["train_non_person"], eval_tf)
    val_ds   = VWWDataset(mp["val_person"],   mp["val_non_person"],   eval_tf)
    def collect(ds, n):
        loader = DataLoader(ds, batch_size=1, shuffle=False)
        xs, ys = [], []
        for i, (x, y) in enumerate(loader):
            if i >= n: break
            xs.append(x.numpy().astype("float32")[0])
            ys.append(int(y.item()))
        return np.stack(xs), np.array(ys, dtype="int32")
    calib_x, _     = collect(train_ds, n_calib)
    val_x,   val_y = collect(val_ds,   n_eval)
    return calib_x, val_x, val_y

class CalibReader(CalibrationDataReader):
    def __init__(self, x, name="input"):
        self.x = x; self.name = name; self.i = 0
    def get_next(self):
        if self.i >= len(self.x): return None
        out = {self.name: self.x[self.i:self.i+1]}; self.i += 1; return out
    def rewind(self): self.i = 0

def export_int8_onnx(fp32_path, calib_x, tag):
    path = str(EXPORT_DIR / f"vww_{tag}_int8_qdq.onnx")
    quantize_static(
        model_input=fp32_path, model_output=path,
        calibration_data_reader=CalibReader(calib_x),
        quant_format=QuantFormat.QDQ,
        activation_type=QuantType.QInt8, weight_type=QuantType.QInt8,
        per_channel=True,
    )
    print(f"✅ INT8 QDQ ONNX: {path}")
    return path

def onnx_accuracy(onnx_path, val_x, val_y):
    sess     = ort.InferenceSession(onnx_path, providers=["CPUExecutionProvider"])
    in_name  = sess.get_inputs()[0].name
    out_name = sess.get_outputs()[0].name
    preds    = [np.argmax(sess.run([out_name], {in_name: val_x[i:i+1]})[0][0])
                for i in range(len(val_x))]
    return round((np.array(preds) == val_y).mean() * 100, 2)

def save_arrays(tag, calib_x, val_x, val_y):
    np.savez(str(EXPORT_DIR / f"vww_{tag}_calib.npz"),  input=calib_x)
    np.savez(str(EXPORT_DIR / f"vww_{tag}_val.npz"),    input=val_x)
    np.savez(str(EXPORT_DIR / f"vww_{tag}_labels.npz"), label=val_y)
    print(f"✅ Arrays saved for {tag}")

def stm32_accuracy(labels_npz, outputs_npz, key="c_outputs_1", num_classes=2):
    y   = np.load(labels_npz)["label"].astype("int64")
    raw = np.load(outputs_npz)[key].reshape(len(y), num_classes)
    acc = (np.argmax(raw, 1) == y).mean() * 100
    print(f"STM32 accuracy: {acc:.2f}%  ({len(y)} samples)")
    return round(acc, 2)


In [ ]:
# ── Export both models: FP32 + INT8 ONNX ────────────────────────────
calib_x, val_x, val_y = collect_arrays(CALIB_SAMPLES, EVAL_SAMPLES)

# MobileNetV2
save_arrays("V2_hw", calib_x, val_x, val_y)
fp32_v2 = export_fp32_onnx(model_v2, "V2_hw")
int8_v2 = export_int8_onnx(fp32_v2, calib_x, "V2_hw")

# MobileNetV3
save_arrays("V3_hw", calib_x, val_x, val_y)
fp32_v3 = export_fp32_onnx(model_v3, "V3_hw")
int8_v3 = export_int8_onnx(fp32_v3, calib_x, "V3_hw")


In [ ]:
# ── ONNX Runtime accuracy check ──────────────────────────────────────
v2_fp32_acc = onnx_accuracy(fp32_v2, val_x, val_y)
v2_int8_acc = onnx_accuracy(int8_v2, val_x, val_y)
v3_fp32_acc = onnx_accuracy(fp32_v3, val_x, val_y)
v3_int8_acc = onnx_accuracy(int8_v3, val_x, val_y)

print(f"\nONNX Runtime Results")
print(f"{'Model':30s}  {'FP32':>8}  {'INT8':>8}  {'Drop':>8}")
print(f"{'-'*58}")
print(f"{'MobileNetV2 scratch':30s}  {v2_fp32_acc:>7.2f}%  {v2_int8_acc:>7.2f}%  {v2_fp32_acc-v2_int8_acc:>+7.2f}%")
print(f"{'MobileNetV3 pretrained':30s}  {v3_fp32_acc:>7.2f}%  {v3_int8_acc:>7.2f}%  {v3_fp32_acc-v3_int8_acc:>+7.2f}%")


## STM32 Deployment Instructions

Run both models on the NUCLEO-N657X0-Q using X-CUBE-AI.

**For each model:**
1. Open STM32CubeIDE → X-CUBE-AI
2. Import the INT8 QDQ ONNX file
3. Use the corresponding `vww_*_hw_val.npz` as input batch
4. Run validation → note accuracy
5. Check profiling report → note inference latency (ms)
6. Rename output NPZ and score with `stm32_accuracy()` below

**Files to deploy:**

| Model | ONNX file | Input arrays |
|-------|-----------|-------------|
| MobileNetV2 | `vww_V2_hw_int8_qdq.onnx` | `vww_V2_hw_val.npz` |
| MobileNetV3 | `vww_V3_hw_int8_qdq.onnx` | `vww_V3_hw_val.npz` |


In [ ]:
# ── STM32 results — fill in after hardware runs ──────────────────────

# After STM32 run for V2:
# v2_stm32_acc = stm32_accuracy(
#     str(EXPORT_DIR / "vww_V2_hw_labels.npz"),
#     "network_val_outputs_V2.npz"
# )

# After STM32 run for V3:
# v3_stm32_acc = stm32_accuracy(
#     str(EXPORT_DIR / "vww_V3_hw_labels.npz"),
#     "network_val_outputs_V3.npz"
# )

# Fill these in manually after hardware runs
HW_RESULTS = {
    "MobileNetV2 scratch": {
        "Val Acc FP32 (%)":   round(evaluate(model_v2, val_loader, device)*100, 2),
        "ONNX FP32 (%)":      v2_fp32_acc,
        "ONNX INT8 (%)":      v2_int8_acc,
        "STM32 INT8 (%)":     None,   # fill after hardware run
        "STM32 Latency (ms)": None,   # fill from X-CUBE-AI profiling
        "Size FP32 (MB)":     round(model_size_mb(model_v2), 2),
        "Size INT8 (MB)":     round(model_size_mb(model_v2)/4, 2),
    },
    "MobileNetV3 pretrained": {
        "Val Acc FP32 (%)":   round(evaluate(model_v3, val_loader, device)*100, 2),
        "ONNX FP32 (%)":      v3_fp32_acc,
        "ONNX INT8 (%)":      v3_int8_acc,
        "STM32 INT8 (%)":     None,
        "STM32 Latency (ms)": None,
        "Size FP32 (MB)":     round(model_size_mb(model_v3), 2),
        "Size INT8 (MB)":     round(model_size_mb(model_v3)/4, 2),
    },
}

import pandas as pd
df = pd.DataFrame(HW_RESULTS).T
print("\nSTM32 Hardware Comparison")
print(df.to_string())
print("\n→ Copy HW_RESULTS into notebook 17_Final_Results hardware section")


In [ ]:
# ── Visualization — runs after STM32 results are filled in ───────────
import matplotlib.pyplot as plt

if all(HW_RESULTS[m]["STM32 INT8 (%)"] is not None for m in HW_RESULTS):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4))
    models  = list(HW_RESULTS.keys())
    colors  = ["#4878D0", "#E8735A"]
    short   = ["MobileNetV2", "MobileNetV3"]

    # Accuracy comparison
    ax = axes[0]
    fp32_accs = [HW_RESULTS[m]["Val Acc FP32 (%)"]  for m in models]
    int8_accs = [HW_RESULTS[m]["ONNX INT8 (%)"]     for m in models]
    stm32_accs= [HW_RESULTS[m]["STM32 INT8 (%)"]    for m in models]
    x = range(len(models))
    w = 0.25
    ax.bar([i-w   for i in x], fp32_accs,  w, label="FP32",       color="#aec6e8", edgecolor="white")
    ax.bar([i     for i in x], int8_accs,  w, label="ONNX INT8",  color="#4878D0", edgecolor="white")
    ax.bar([i+w   for i in x], stm32_accs, w, label="STM32 INT8", color="#1a4a8a", edgecolor="white")
    ax.set_xticks(list(x)); ax.set_xticklabels(short)
    ax.set_ylabel("Accuracy (%)"); ax.set_title("Accuracy Across Formats")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.4)

    # Latency comparison
    ax = axes[1]
    lats = [HW_RESULTS[m]["STM32 Latency (ms)"] for m in models]
    bars = ax.bar(short, lats, color=colors, edgecolor="white")
    for b in bars:
        ax.annotate(f"{b.get_height():.1f}ms",
                   (b.get_x()+b.get_width()/2, b.get_height()),
                   ha="center", va="bottom", fontsize=10)
    ax.set_ylabel("Latency (ms)"); ax.set_title("STM32 Inference Latency")
    ax.grid(axis="y", alpha=0.4)

    # Size comparison
    ax = axes[2]
    fp32_sizes = [HW_RESULTS[m]["Size FP32 (MB)"] for m in models]
    int8_sizes = [HW_RESULTS[m]["Size INT8 (MB)"] for m in models]
    ax.bar([i-w/2 for i in x], fp32_sizes, w*2, label="FP32", color="#aec6e8", edgecolor="white")
    ax.bar([i+w/2 for i in x], int8_sizes, w*2, label="INT8", color="#4878D0", edgecolor="white")
    ax.set_xticks(list(x)); ax.set_xticklabels(short)
    ax.set_ylabel("Size (MB)"); ax.set_title("Model Size FP32 vs INT8")
    ax.legend(fontsize=8); ax.grid(axis="y", alpha=0.4)

    plt.suptitle("MobileNetV2 vs MobileNetV3 — STM32 Hardware Comparison", fontsize=12, y=1.02)
    plt.tight_layout(); plt.show()
else:
    print("Fill in STM32 INT8 accuracy and latency in HW_RESULTS to generate plots.")
